In [ ]:
from datasets import load_dataset, concatenate_datasets
import pandas as pd

# Ссылки на raw CSV-файлы Hugging Face
url_nvinder = "https://huggingface.co/datasets/shnyakov/nenets/raw/main/articles_nvinder.csv"
url_tales   = "https://huggingface.co/datasets/shnyakov/nenets/raw/main/parallel_corpus_tales_yangasova.csv"

# 1. Загрузка datasets
nv = load_dataset("csv", data_files={"train": url_nvinder})["train"]
tales_trans = load_dataset("csv", data_files={"train": url_tales})["train"]
tales_article = load_dataset("csv", data_files={"train": url_tales})["train"]

# 2. Нормализация колонок
nv = nv.rename_column("example_ru", "ru").rename_column("example_nn", "nn")
nv = nv.remove_columns(["source"])

tales_trans = tales_trans.rename_column("translate_ru", "ru").rename_column("translate_nn", "nn")
tales_trans = tales_trans.remove_columns(["source", "article_ru", "article_nn"])

tales_article = tales_article.rename_column("article_ru", "ru").rename_column("article_nn", "nn")
tales_article = tales_article.remove_columns(["source", "translate_ru", "translate_nn"])

# 3. Объединение и очистка
combined = concatenate_datasets([nv, tales_trans, tales_article])
df = combined.to_pandas()
# Удаляем пустые и дубли
df = df.dropna(subset=["ru", "nn"])
df = df[df["ru"].str.strip() != ""]
df = df[df["nn"].str.strip() != ""]
df = df.drop_duplicates(subset=["ru", "nn"])
# Возвращаем Dataset
from datasets import Dataset
combined_clean = Dataset.from_pandas(df, preserve_index=False)

# 4. Разбиение на train/validation
split = combined_clean.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
val_dataset = split["test"]

print(f"Всего примеров: {len(combined_clean)}")
print(f"Train: {len(train_dataset)}, Validation: {len(val_dataset)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Всего примеров: 3257
Train: 2931, Validation: 326


In [ ]:
# Сохранение обучающих текстов для токенизатора
with open("train_ru.txt", "w") as f_ru, open("train_nn.txt", "w") as f_nn:
    for row in train_dataset:
        f_ru.write(row["ru"] + "\n")
        f_nn.write(row["nn"] + "\n")

In [ ]:
!pip install tokenizers transformers

from tokenizers import ByteLevelBPETokenizer
# Создаем пустой токенизатор и обучаем его на текстах русского и ненецкого
tokenizer = ByteLevelBPETokenizer()
tokenizer.train(files=["train_ru.txt", "train_nn.txt"],
                vocab_size=16000, min_frequency=2,
                special_tokens=["<bos>", "<eos>", "<pad>", "<unk>"])
# Сохраняем обученный токенизатор
tokenizer.save("nenets_tokenizer.json")

In [ ]:
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="nenets_tokenizer.json",
    bos_token="<bos>",
    eos_token="<eos>",
    pad_token="<pad>",
    unk_token="<unk>",
)

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")

model.resize_token_embeddings(len(tokenizer))


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Embedding(16000, 512)

In [ ]:
def preprocess_function(examples):
    return tokenizer(
        examples["ru"],
        text_target=examples["nn"],
        max_length=128,
        truncation=True
    )

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val  = val_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/2931 [00:00<?, ? examples/s]

Map:   0%|          | 0/326 [00:00<?, ? examples/s]

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

args = Seq2SeqTrainingArguments(
    output_dir="ru-nn-mt5",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=collator
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 0, 'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shnyakov to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,No log,10.638534
2,15.626600,9.366363
3,10.922000,9.092611


TrainOutput(global_step=1101, training_loss=13.006729278425864, metrics={'train_runtime': 209.8066, 'train_samples_per_second': 41.91, 'train_steps_per_second': 5.248, 'total_flos': 219205299793920.0, 'train_loss': 13.006729278425864, 'epoch': 3.0})

In [ ]:
def encode_for_mt5(texts):
    batch = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    # remove token_type_ids
    return {k: v for k, v in batch.items() if k != "token_type_ids"}

In [ ]:
model.eval()
device = model.device  # usually cuda:0

test_sentences = [
    "Привет, как дела?",
    "Я иду домой.",
    "Спасибо за помощь."
]

references = ["-", "-", "-"]  # placeholder

# 1. токенизация
inputs = tokenizer(
    test_sentences,
    return_tensors="pt",
    padding=True,
    truncation=True
)

# 2. удаляем token_type_ids
inputs = {k: v for k, v in inputs.items() if k != "token_type_ids"}

# 3. перенос на GPU
inputs = {k: v.to(device) for k, v in inputs.items()}

# 4. генерация
outputs = model.generate(
    **inputs,
    max_length=150
)

# 5. декодирование
predictions = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

# 6. вывод
for ru, pred, ref in zip(test_sentences, predictions, references):
    print(f"RU: {ru}\nMODEL: {pred}\nREFERENCE: {ref}\n")


RU: Привет, как дела?
MODEL: Мастер’�’’�’’�’’�’’�’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’
REFERENCE: -

RU: Я иду домой.
MODEL:  Мужчины’’Я’Я’Я’Я’Я’Я’Я’Я’’Я’Я’’ЯЯ’’ЯЯ’’ЯЯ’’ЯЯ’’ЯЯ’’ЯЯ’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я’Я
REFERENCE: -

RU: Спасибо за помощь.
MODEL:  Нись’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’
REFERENCE: -

